In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

DataFrame[]

#1) silver.tb_info_filmes

Normalização de status, datas multi-formato, tipagem segura
e unicidade por filme priorizando a ingestão mais recente.

In [0]:
df = spark.table("workspace.bronze.tb_movies_info")

status_norm = F.lower(
    F.trim(
        F.regexp_replace(
            F.regexp_replace(F.col("status"), r"[-_]+", " "),
            r"\s+", " "
        )
    )
)

df = (
    df
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
            F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
            F.expr("try_to_date(release_date, 'MM/dd/yyyy')"),
            F.expr("try_to_date(release_date, 'yyyy/MM/dd')"),
            F.expr("try_to_date(release_date, 'dd-MM-yyyy')"),
            F.expr("try_to_date(release_date, 'MM-dd-yyyy')")
        )
    )
    .withColumn(
        "duracao_minutos",
        F.expr("try_cast(runtime as int)")
    )
    .withColumn(
        "status_filme",
        F.when(status_norm == "released", "Lançado")
         .when(status_norm == "post production", "Pós-Produção")
         .when(status_norm == "in production", "Em Produção")
         .when(status_norm == "planned", "Planejado")
         .when(status_norm == "rumored", "Rumores")
         .when(status_norm == "canceled", "Cancelado")
         .otherwise("Não Informado")
    )
)

# Em caso de empate na data de ingestão, prioriza o registro
# com maior quantidade de dados úteis.
df = df.withColumn(
    "_qualidade",
    F.when(F.col("data_lancamento").isNotNull(), 1).otherwise(0) +
    F.when(F.col("duracao_minutos").isNotNull(), 1).otherwise(0) +
    F.when(F.col("overview").isNotNull(), 1).otherwise(0) +
    F.when(F.col("tagline").isNotNull(), 1).otherwise(0)
)

janela = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc(),
    F.col("_qualidade").desc()
)

df_info_silver = (
    df
    .withColumn("_rn", F.row_number().over(janela))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("id").alias("id_filme"),
        F.col("title").alias("titulo"),
        F.col("original_title").alias("titulo_original"),
        "data_lancamento",
        F.year("data_lancamento").alias("ano_lancamento"),
        "duracao_minutos",
        F.col("original_language").alias("idioma_original"),
        "status_filme",
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao")
    )
)

assert (
    df_info_silver.count()
    == df_info_silver.select("id_filme").distinct().count()
), "tb_info_filmes possui IDs duplicados"

(
    df_info_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_info_filmes")
)

print("✓ silver.tb_info_filmes concluída")

✓ silver.tb_info_filmes concluída


#2) silver.tb_financeiro_filmes

Limpeza financeira, tratamento de K/M/B, deduplicação,
conversão USD -> BRL e cálculo de lucro/margem.


In [0]:
df = spark.table("workspace.bronze.tb_movies_financials")

def dinheiro(coluna):
    texto = F.upper(F.trim(F.col(coluna)))

    numero = F.expr(f"""
        try_cast(
            regexp_extract(
                regexp_replace(upper(trim({coluna})), ',', ''),
                '(-?[0-9]+(?:\\\\.[0-9]+)?)',
                1
            )
            as decimal(20,4)
        )
    """)

    valor = (
        F.when(texto.rlike(r"K\s*$"), numero * 1_000)
         .when(texto.rlike(r"M\s*$"), numero * 1_000_000)
         .when(texto.rlike(r"B\s*$"), numero * 1_000_000_000)
         .otherwise(numero)
    )

    return F.when(valor > 0, valor)


df = (
    df
    .withColumn("orcamento_usd", dinheiro("budget"))
    .withColumn("receita_usd", dinheiro("revenue"))
)

# Prioriza o registro mais recente e com maior quantidade
# de informações financeiras válidas.
df = df.withColumn(
    "_qualidade",
    F.when(F.col("orcamento_usd").isNotNull(), 1).otherwise(0) +
    F.when(F.col("receita_usd").isNotNull(), 1).otherwise(0)
)

# O hash serve apenas como desempate determinístico caso
# data de ingestão e qualidade sejam iguais.
janela = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc(),
    F.col("_qualidade").desc(),
    F.xxhash64("budget", "revenue").desc()
)

df = (
    df
    .withColumn("_rn", F.row_number().over(janela))
    .filter(F.col("_rn") == 1)
)

# Utiliza a cotação mais recente disponível na Bronze.
cotacao_atual = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .select(F.col("cotacaoCompra").cast("decimal(10,4)"))
    .first()[0]
)

df_fin_silver = (
    df
    .select(
        F.col("id").alias("id_filme"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)")
    )
    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * F.lit(cotacao_atual))
        .cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * F.lit(cotacao_atual))
        .cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_usd",
        (F.col("receita_usd") - F.col("orcamento_usd"))
        .cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (F.col("receita_brl") - F.col("orcamento_brl"))
        .cast("decimal(18,2)")
    )
    .withColumn(
        "margem_lucro_percentual",
        (
            (F.col("receita_usd") - F.col("orcamento_usd"))
            / F.col("orcamento_usd") * 100
        ).cast("decimal(18,2)")
    )
)

assert (
    df_fin_silver.count()
    == df_fin_silver.select("id_filme").distinct().count()
), "tb_financeiro_filmes possui IDs duplicados"

(
    df_fin_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.tb_financeiro_filmes")
)

print(
    f"✓ silver.tb_financeiro_filmes concluída | "
    f"Cotação USD/BRL: {cotacao_atual}"
)

✓ silver.tb_financeiro_filmes concluída | Cotação USD/BRL: 5.1569


#3) silver.tb_metricas_engajamento

Conversão segura por causa de Column Shift, tratamento de separadores
e aplicação das regras de faixa/negatividade.


In [0]:
df = spark.table("workspace.bronze.tb_movies_metrics")

df = (
    df
    .withColumn(
        "popularidade",
        F.expr("""
            try_cast(
                regexp_replace(trim(popularity), ',', '.')
                as double
            )
        """)
    )
    .withColumn(
        "nota_media_tmdb",
        F.expr("try_cast(vote_average as double)")
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.expr("try_cast(vote_count as int)")
    )
    .withColumn(
        "nota_media_imdb",
        F.expr("try_cast(averageRating as double)")
    )
    .withColumn(
        "qtd_votos_imdb",
        F.expr("try_cast(numVotes as int)")
    )
)

df = (
    df
    .withColumn(
        "popularidade",
        F.when(F.col("popularidade") >= 0, F.col("popularidade"))
    )
    .withColumn(
        "nota_media_tmdb",
        F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb"))
    )
    .withColumn(
        "nota_media_imdb",
        F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb"))
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb"))
    )
    .withColumn(
        "qtd_votos_imdb",
        F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb"))
    )
)

df = df.withColumn(
    "_qualidade",
    F.when(F.col("popularidade").isNotNull(), 1).otherwise(0) +
    F.when(F.col("nota_media_tmdb").isNotNull(), 1).otherwise(0) +
    F.when(F.col("qtd_votos_tmdb").isNotNull(), 1).otherwise(0) +
    F.when(F.col("nota_media_imdb").isNotNull(), 1).otherwise(0) +
    F.when(F.col("qtd_votos_imdb").isNotNull(), 1).otherwise(0)
)

janela = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc(),
    F.col("_qualidade").desc()
)

df_metricas_silver = (
    df
    .withColumn("_rn", F.row_number().over(janela))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("id").alias("id_filme"),
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

assert (
    df_metricas_silver.count()
    == df_metricas_silver.select("id_filme").distinct().count()
), "tb_metricas_engajamento possui IDs duplicados"

(
    df_metricas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.tb_metricas_engajamento")
)

print("✓ silver.tb_metricas_engajamento concluída")

✓ silver.tb_metricas_engajamento concluída


#4) silver.tb_avaliacoes_usuarios 

Remove avaliações duplicadas, valida notas de 0 a 10
e padroniza comentários ausentes.

In [0]:
df = (
    spark.table("workspace.bronze.tb_movies_reviews")

    # A duplicidade é definida pelos valores originais da avaliação.
    .dropDuplicates(["id", "nome", "nota", "comentario"])

    .select(
        F.col("id").alias("id_filme"),
        F.col("nome").alias("nome_usuario"),

        F.when(
            F.expr("try_cast(nota as double)").between(0, 10),
            F.expr("try_cast(nota as double)")
        ).alias("nota_usuario"),

        F.when(
            F.col("comentario").isNull() |
            (F.trim(F.col("comentario")) == ""),
            "Sem comentário"
        )
        .otherwise(F.trim(F.col("comentario")))
        .alias("comentario_usuario")
    )
)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.tb_avaliacoes_usuarios")
)

print("✓ silver.tb_avaliacoes_usuarios concluída")

✓ silver.tb_avaliacoes_usuarios concluída


#5) silver.tb_generos

Separa os múltiplos gêneros em linhas individuais e remove
resíduos inválidos causados por separadores inconsistentes/Column Shift.

In [0]:
# Filtra apenas gêneros válidos para remover resíduos de Column Shift da coluna genres.

generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War", "Western"
]

df_generos = (
    spark.table("workspace.bronze.tb_credits_and_tags")
    .withColumn(
        "genero",
        F.explode(
            F.split(
                F.regexp_replace(F.col("genres"), r"[;|]", ","),
                ","
            )
        )
    )
    .withColumn("genero", F.trim(F.col("genero")))
    .filter(F.col("genero").isin(generos_validos))
    .select(
        F.col("id").alias("id_filme"),
        F.col("genero").alias("nome_genero")
    )
    .dropDuplicates(["id_filme", "nome_genero"])
)

(
    df_generos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.tb_generos")
)

print("✓ silver.tb_generos concluída")

✓ silver.tb_generos concluída


#6) silver.tb_pessoas_empresas

Consolida atores, diretores, roteiristas e produtoras em uma única tabela.%md


In [0]:
df_origem = spark.table("workspace.bronze.tb_credits_and_tags")

df_entidades = (
    df_origem
    .select(
        "id",
        F.explode(
            F.array(
                F.struct(F.col("cast").alias("valores"), F.lit("Ator").alias("tipo_entidade")),
                F.struct(F.col("directors").alias("valores"), F.lit("Diretor").alias("tipo_entidade")),
                F.struct(F.col("writers").alias("valores"), F.lit("Roteirista").alias("tipo_entidade")),
                F.struct(F.col("production_companies").alias("valores"), F.lit("Produtora").alias("tipo_entidade"))
            )
        ).alias("entidade")
    )
    .select(
        "id",
        F.col("entidade.valores").alias("valores"),
        F.col("entidade.tipo_entidade").alias("tipo_entidade")
    )

    # Normaliza vírgula, ponto e vírgula e "|" como separadores.
    .withColumn(
        "nome_entidade",
        F.explode(
            F.split(
                F.regexp_replace(F.col("valores"), r"[;|]", ","),
                ","
            )
        )
    )

    .withColumn(
        "nome_entidade",
        F.initcap(F.trim(F.col("nome_entidade")))
    )

    # Remove vazios, placeholders, números puros, caminhos de imagem
    # e resíduos excessivamente longos causados por Column Shift.
    .filter(
        F.col("nome_entidade").isNotNull() &
        (F.col("nome_entidade") != "") &
        (~F.col("nome_entidade").rlike(r"^\d+(\.\d+)?$")) &
        (~F.upper(F.col("nome_entidade")).isin(
            "[]", "N/A", "UNKNOWN", "NÃO INFORMADO", "NULL", "NONE"
        )) &
        (~F.lower(F.col("nome_entidade")).rlike(
            r"^/|.*\.(jpg|jpeg|png|webp)$"
        )) &
        (F.length(F.col("nome_entidade")) <= 100)
    )

    .select(
        F.col("id").alias("id_filme"),
        "nome_entidade",
        "tipo_entidade"
    )

    .dropDuplicates(
        ["id_filme", "nome_entidade", "tipo_entidade"]
    )
)

(
    df_entidades.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.tb_pessoas_empresas")
)

print("✓ silver.tb_pessoas_empresas concluída")

✓ silver.tb_pessoas_empresas concluída


#7) silver.tb_cotacao_dolar

Mantém uma cotação por dia e preenche dias sem cotação com o último valor disponível (Forward Fill).

In [0]:
df_cotacao = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .withColumn(
        "data_hora_cotacao",
        F.to_timestamp("dataHoraCotacao")
    )
    .withColumn(
        "data_cotacao",
        F.to_date("data_hora_cotacao")
    )
    .withColumn(
        "cotacao_compra",
        F.col("cotacaoCompra").cast("decimal(10,4)")
    )
)

# Caso existam várias ingestões/cotações no mesmo dia,
# mantém apenas a mais recente.
janela_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("data_hora_cotacao").desc(),
    F.col("ingestion_datetime").desc()
)

df_cotacao = (
    df_cotacao
    .withColumn("_rn", F.row_number().over(janela_dia))
    .filter(F.col("_rn") == 1)
    .select("data_cotacao", "cotacao_compra")
)

# O calendário vai da primeira cotação conhecida até hoje,
# permitindo Forward Fill também em fins de semana e feriados.
limites = (
    df_cotacao
    .agg(F.min("data_cotacao").alias("inicio"))
    .withColumn("fim", F.current_date())
)

calendario = (
    limites
    .select(
        F.explode(
            F.sequence(
                F.col("inicio"),
                F.col("fim"),
                F.expr("interval 1 day")
            )
        ).alias("data_cotacao")
    )
)

janela_ffill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_silver = (
    calendario
    .join(df_cotacao, "data_cotacao", "left")
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(janela_ffill)
    )
)

(
    df_cotacao_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.tb_cotacao_dolar")
)

print("✓ silver.tb_cotacao_dolar concluída")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ silver.tb_cotacao_dolar concluída


#8) Validação final da camada Silver

In [0]:
tabelas_silver = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

for tabela in tabelas_silver:
    nome = f"workspace.silver.{tabela}"
    qtd = spark.table(nome).count()

    assert qtd > 0, f"{tabela} está vazia"

    print(f"✓ {tabela}: {qtd:,} registros")

print("\n✓ Camada Silver concluída com sucesso")

✓ tb_info_filmes: 97,879 registros
✓ tb_financeiro_filmes: 99,006 registros
✓ tb_metricas_engajamento: 99,013 registros
✓ tb_avaliacoes_usuarios: 32,412 registros
✓ tb_generos: 141,965 registros
✓ tb_pessoas_empresas: 896,754 registros
✓ tb_cotacao_dolar: 9 registros

✓ Camada Silver concluída com sucesso
